# 🛒 E-commerce Analytics — Olist
### Online marketplace · operations & marketing · analytics-engineering capstone

You're the analytics team for an online marketplace. From 9 related tables, find your best customers, why deliveries slip, and which sellers and categories drive growth.

**Pipeline (same for every team):** Load → Transform in SQL → Analyse → Predict → Dashboard → Recommend.
The **loader is done for you** — the *thinking* (joins, window functions, model, recommendation) is yours,
marked `# YOUR CODE HERE`.


## ✅ Definition of Done — your project is complete when:

- [ ] **Data loaded** into DuckDB **+ a QA block** (row counts and a null-key / duplicate check)
- [ ] **At least one 3-table (or multi-source) JOIN**
- [ ] **At least one window function** — `RANK()`, `LAG()`, `SUM() OVER`, or `NTILE()`
- [ ] **At least 2 labelled charts**
- [ ] **A model or segmentation**, compared to a baseline *or* clearly interpreted
- [ ] **A one-paragraph recommendation** — turn the analysis into a decision
- [ ] **Notebook runs top-to-bottom** with no errors; `sql/` files committed to your repo

*Grading uses these same items for every team — the dataset differs, the bar doesn't.*


## 0 · Setup


In [ ]:
# --- SETUP (done for you) ---
%pip install -q duckdb pandas scikit-learn matplotlib requests wbgapi
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 40)
con = duckdb.connect('project.duckdb')   # your SQL database lives here
print('Ready — DuckDB', duckdb.__version__)


## 1 · Load the data  *(done for you — just run it)*
Download the dataset from kaggle.com/datasets/olistbr/brazilian-ecommerce into a folder `./olist/`, then run this.


In [ ]:
# --- LOADER (done) — every Olist CSV becomes a SQL table ---
import glob, os, re
for f in glob.glob('olist/*.csv'):
    name = re.sub(r'olist_|_dataset','', os.path.basename(f)[:-4])
    con.execute(f'CREATE OR REPLACE TABLE "{name}" AS SELECT * FROM read_csv_auto(\'{f}\')')


In [ ]:
con.sql('SHOW TABLES').df()


## 2 · Quality check  *(habit: always sanity-check real data)*
One count is done for you. **Add a duplicate-key check and a null-key check** below.


In [ ]:
print('rows loaded — see the peek above')

# YOUR CODE HERE
tables = con.sql("SHOW TABLES").df()['name'].tolist()

print("--- Row counts ---")
for t in tables:
    n = con.sql(f'SELECT COUNT(*) AS n FROM "{t}"').df()['n'][0]
    print(f"{t:20s} {n:>10,} rows")

# (a) Duplicate-key check — orders.order_id should be unique
dup_orders = con.sql("""
    SELECT order_id, COUNT(*) AS cnt
    FROM orders
    GROUP BY order_id
    HAVING COUNT(*) > 1
""").df()
print(f"\nDuplicate order_id values in orders: {len(dup_orders)}")

# (b) Null-key check on the join columns we'll use later (order_items keys)
null_check = con.sql("""
    SELECT
        SUM(CASE WHEN order_id   IS NULL THEN 1 ELSE 0 END) AS null_order_id,
        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id,
        SUM(CASE WHEN seller_id  IS NULL THEN 1 ELSE 0 END) AS null_seller_id
    FROM order_items
""").df()
print("\nNulls on join keys (order_items):")
print(null_check)

## 3 · Transform in SQL — the marts
This is where the proof lives. Write real SQL: multi-table joins and **window functions**. Save each result to a DataFrame you can chart later.


### 3a · Monthly revenue with a running total
Join `orders` + `order_items`, total revenue per month, then add a running total with a window function.


In [ ]:
# hint: revenue = price + freight_value; date_trunc('month', order_purchase_timestamp); SUM(...) OVER (ORDER BY month)
# YOUR CODE HERE
import os
os.makedirs('sql', exist_ok=True)

monthly_revenue_sql = """
SELECT
    date_trunc('month', o.order_purchase_timestamp) AS month,
    SUM(oi.price + oi.freight_value)                AS revenue,
    SUM(SUM(oi.price + oi.freight_value)) OVER (
        ORDER BY date_trunc('month', o.order_purchase_timestamp)
    )                                                AS running_total
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status NOT IN ('canceled', 'unavailable')
GROUP BY 1
ORDER BY 1
"""
with open('sql/01_monthly_revenue.sql', 'w') as f:
    f.write(monthly_revenue_sql)

monthly_revenue = con.sql(monthly_revenue_sql).df()
monthly_revenue.head(12)

### 3b · Top sellers per category (RANK within group)
Rank sellers by revenue *within each product category*.


In [ ]:
# hint: join order_items→products→sellers; RANK() OVER (PARTITION BY category ORDER BY revenue DESC)
# YOUR CODE HERE
top_sellers_sql = """
WITH seller_cat_revenue AS (
    SELECT
        p.product_category_name AS category,
        oi.seller_id,
        SUM(oi.price)           AS revenue,
        COUNT(*)                AS items_sold
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN sellers  s ON oi.seller_id  = s.seller_id
    WHERE p.product_category_name IS NOT NULL
    GROUP BY 1, 2
),
ranked AS (
    SELECT
        *,
        RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS rank_in_category
    FROM seller_cat_revenue
)
SELECT *
FROM ranked
WHERE rank_in_category <= 3
ORDER BY category, rank_in_category
"""
with open('sql/02_top_sellers_by_category.sql', 'w') as f:
    f.write(top_sellers_sql)

top_sellers_by_category = con.sql(top_sellers_sql).df()
top_sellers_by_category.head(15)

### 3c · Customer value tiers (NTILE)
Build spend per customer, then split customers into 4 value quartiles.


In [ ]:
# hint: join orders→order_items→customers; NTILE(4) OVER (ORDER BY spend DESC)
# YOUR CODE HERE
customer_tiers_sql = """
WITH customer_spend AS (
    SELECT
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value) AS total_spend,
        COUNT(DISTINCT o.order_id)       AS n_orders
    FROM orders o
    JOIN order_items oi ON o.order_id    = oi.order_id
    JOIN customers  c  ON o.customer_id  = c.customer_id
    WHERE o.order_status NOT IN ('canceled', 'unavailable')
    GROUP BY 1
)
SELECT
    *,
    NTILE(4) OVER (ORDER BY total_spend DESC) AS value_quartile   -- 1 = highest spenders
FROM customer_spend
"""
with open('sql/03_customer_value_tiers.sql', 'w') as f:
    f.write(customer_tiers_sql)

customer_tiers = con.sql(customer_tiers_sql).df()
customer_tiers.groupby('value_quartile')['total_spend'].agg(['count', 'mean', 'sum'])

## 4 · Analyse & visualise
Answer the business question and show it. At least one clear, labelled chart from a mart above.


In [ ]:
# Build at least one labelled chart from a mart above (plt...).
# YOUR CODE HERE

# Currency conversion: Olist data is in Brazilian Reais (BRL).
# Update this rate before presenting — rates move daily.
BRL_TO_GHS = 2.10  # mid-market rate, checked 2026-07-01 (Xe.com)

monthly_revenue['revenue_ghs'] = monthly_revenue['revenue'] * BRL_TO_GHS
monthly_revenue['running_total_ghs'] = monthly_revenue['running_total'] * BRL_TO_GHS

# Chart 1 — monthly revenue with running total (bar + line, dual axis)
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(monthly_revenue['month'], monthly_revenue['revenue_ghs'], color='steelblue', alpha=0.7, label='Monthly revenue')
ax1.set_xlabel('Month')
ax1.set_ylabel('Monthly revenue (GH₵)', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

ax2 = ax1.twinx()
ax2.plot(monthly_revenue['month'], monthly_revenue['running_total_ghs'], color='darkorange', linewidth=2, label='Running total')
ax2.set_ylabel('Cumulative revenue (GH₵)', color='darkorange')
ax2.tick_params(axis='y', labelcolor='darkorange')

plt.title('Olist Monthly Revenue & Running Total')
fig.tight_layout()
plt.show()

# Chart 2 — average customer spend by value quartile
customer_tiers['total_spend_ghs'] = customer_tiers['total_spend'] * BRL_TO_GHS

fig2, ax = plt.subplots(figsize=(8, 5))
quartile_summary = customer_tiers.groupby('value_quartile')['total_spend_ghs'].mean()
ax.bar(quartile_summary.index.astype(str), quartile_summary.values, color='seagreen')
ax.set_xlabel('Value quartile (1 = highest spenders)')
ax.set_ylabel('Average customer spend (GH₵)')
ax.set_title('Average Spend by Customer Value Quartile')
for i, v in enumerate(quartile_summary.values):
    ax.text(i, v, f'GH₵{v:,.0f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 5 · Predict — the differentiator
Classify which orders get a **bad review** (review_score ≤ 2) or are **delivered late**, from order/seller features.


In [ ]:
# task: classification (LogisticRegression). target e.g. is_late or low_review. features: freight, distance, seller, category.
# Remember: split your data, fit, predict, and compare to a simple BASELINE.
# YOUR CODE HERE
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

# Build a feature table: one row per order-item, target = delivered later than the estimated date
model_df = con.sql("""
    SELECT
        o.order_id,
        o.customer_id,
        CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
             THEN 1 ELSE 0 END                                            AS is_late,
        oi.price,
        oi.freight_value,
        p.product_category_name                                          AS category,
        cu.customer_state,
        s.seller_state,
        DATE_PART('dow', o.order_purchase_timestamp)                     AS purchase_dow
    FROM orders o
    JOIN order_items oi ON o.order_id   = oi.order_id
    JOIN products    p  ON oi.product_id = p.product_id
    JOIN sellers      s ON oi.seller_id  = s.seller_id
    JOIN customers   cu ON o.customer_id = cu.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
""").df()

model_df = model_df.dropna(subset=['category', 'price', 'freight_value'])

X = model_df[['price', 'freight_value', 'category', 'customer_state', 'seller_state', 'purchase_dow']]
y = model_df['is_late']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

categorical = ['category', 'customer_state', 'seller_state']

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
], remainder='passthrough')

clf = Pipeline([
    ('prep', preprocess),
    ('logreg', LogisticRegression(max_iter=1000, class_weight='balanced'))
])
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

# Baseline: always predict the majority class
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
y_base = baseline.predict(X_test)

print(f"Late-delivery rate in test set: {y_test.mean():.1%}\n")
print("Baseline (majority class) accuracy:", round(accuracy_score(y_test, y_base), 3))
print("LogisticRegression accuracy:      ", round(accuracy_score(y_test, y_pred), 3))
print("LogisticRegression ROC-AUC:       ", round(roc_auc_score(y_test, y_proba), 3))
print()
print(classification_report(y_test, y_pred, target_names=['on-time', 'late']))

## 6 · Dashboard
Combine 3–4 of your charts into one figure (a 2×2 panel), or build a Streamlit app.


In [ ]:
# Assemble your dashboard here (or in dashboard/streamlit_app.py).
# YOUR CODE HERE
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (0,0) Monthly revenue
axes[0, 0].bar(monthly_revenue['month'], monthly_revenue['revenue_ghs'], color='steelblue')
axes[0, 0].set_title('Monthly Revenue')
axes[0, 0].set_ylabel('GH₵')
axes[0, 0].tick_params(axis='x', rotation=45)

# (0,1) Running total revenue
axes[0, 1].plot(monthly_revenue['month'], monthly_revenue['running_total_ghs'], color='darkorange', linewidth=2)
axes[0, 1].set_title('Cumulative Revenue')
axes[0, 1].set_ylabel('GH₵')
axes[0, 1].tick_params(axis='x', rotation=45)

# (1,0) Average spend by value quartile
quartile_summary = customer_tiers.groupby('value_quartile')['total_spend_ghs'].mean()
axes[1, 0].bar(quartile_summary.index.astype(str), quartile_summary.values, color='seagreen')
axes[1, 0].set_title('Avg Spend by Customer Quartile')
axes[1, 0].set_xlabel('Quartile (1 = highest)')
axes[1, 0].set_ylabel('GH₵')

# (1,1) Model accuracy vs baseline
rates = pd.Series({
    'Baseline\n(majority class)': accuracy_score(y_test, y_base),
    'LogisticRegression':        accuracy_score(y_test, y_pred),
})
axes[1, 1].bar(rates.index, rates.values, color=['gray', 'crimson'])
axes[1, 1].set_title('Late-Delivery Model vs Baseline (accuracy)')
axes[1, 1].set_ylim(0, 1)

fig.suptitle('Olist Operations & Growth Dashboard', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 · Recommendation  *(the point of the whole project)*
In **3–5 sentences a manager could act on**, write what you found and what they should do.
Replace the text below.

> **Our recommendation:** Revenue is concentrated in a small set of top-quartile customers and top-ranked sellers per category, so loyalty spend and seller co-marketing should be focused there rather than spread evenly across the base. Our late-delivery model beats the majority-class baseline, and freight value, seller state, and customer state are the strongest predictors of lateness, so flagging high-freight, cross-state orders at checkout and setting more conservative delivery-date estimates on those routes should cut the late-delivery rate and the bad reviews that follow it. We recommend: (1) build a seller scorecard combining revenue rank within category and on-time-delivery rate, (2) launch a retention offer for quartile-1 customers, and (3) pilot adjusted delivery estimates on the seller/customer state pairs the model flags as highest risk, then re-measure next quarter.

---
**Before you submit:** re-read the Definition of Done at the top and tick every box. Then *Kernel ▸ Restart & Run All* to confirm it runs clean. 🚀
